# PSELDNets — Colab セットアップ（v4: 途中保存＆再開対応）

## ⚠️ 使用前に必ず確認

1. **ランタイム → ランタイムのタイプを変更 → T4 GPU** を選択
2. セルを **上から順に** 実行（途中でスキップしない）
3. 切断したら、再接続して **また上から順に実行**。学習は Drive の last.ckpt から自動で続きから再開します

---
## 1. GPU 確認

In [ ]:
import torch
assert torch.cuda.is_available(), '⚠️ GPU未接続。ランタイムのタイプを T4 GPU に変更してください。'
print(f'PyTorch : {torch.__version__}')
print(f'GPU     : {torch.cuda.get_device_name(0)}')
print(f'VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 2. リポジトリ clone

In [ ]:
import os

REPO = '/content/PSELDNets'

if not os.path.exists(f'{REPO}/src'):
    !git clone https://github.com/Jinbo-Hu/PSELDNets {REPO}
else:
    print(f'既にあります: {REPO}')

os.chdir(REPO)
print(f'CWD: {os.getcwd()}')
!ls

## 3. パッケージインストール

`numpy` / `h5py` / `scipy` / `torch` は Colab に最初から入っています。
ここでは**触らず**、不足しているものだけ追加します（再起動不要）。

In [ ]:
!pip install -q \
    librosa \
    soundfile \
    lightning==2.2.1 \
    hydra-core==1.3.2 \
    hydra-colorlog==1.2.0 \
    hydra-joblib-launcher==1.2.0 \
    torchmetrics==1.3.1

print('完了')

## 4. 動作確認

In [ ]:
import numpy as np, h5py, lightning, torchmetrics, librosa
print(f'numpy       : {np.__version__}')
print(f'h5py        : {h5py.__version__}')
print(f'lightning   : {lightning.__version__}')
print(f'torchmetrics: {torchmetrics.__version__}')
print(f'librosa     : {librosa.__version__}')
print('すべて OK')

---
## 5. チェックポイントのダウンロード

In [ ]:
import shutil
from huggingface_hub import hf_hub_download

os.makedirs('ckpts', exist_ok=True)
CKPT = 'ckpts/mACCDOA-HTSAT-0.567.ckpt'

if not os.path.exists(CKPT):
    print('ダウンロード中...')
    src = hf_hub_download(
        repo_id='Jinbo-HU/PSELDNets',
        filename='model/mACCDOA-HTSAT-0.567.ckpt',
        repo_type='dataset',
    )
    shutil.copy(src, CKPT)

print(f'OK: {CKPT}  ({os.path.getsize(CKPT)/1e6:.0f} MB)')

## 6. クラス定義ファイル (TSV)

In [ ]:
os.makedirs('datasets', exist_ok=True)

for tsv in ['cls_indices_train.tsv', 'cls_indices_test.tsv']:
    dest = f'datasets/{tsv}'
    if not os.path.exists(dest):
        src = hf_hub_download(
            repo_id='Jinbo-HU/PSELDNets',
            filename=f'dataset/{tsv}',
            repo_type='dataset',
        )
        shutil.copy(src, dest)
    print(f'OK: {dest}')

## 7. テストデータのダウンロード（約4GB）

In [ ]:
import zipfile

ZIP = 'datasets/test360_ov3.zip'
DATA_DIR = 'datasets/test360_ov3'

if not os.path.exists(f'{DATA_DIR}/foa'):
    if not os.path.exists(ZIP):
        print('ダウンロード中（約4GB）...')
        src = hf_hub_download(
            repo_id='Jinbo-HU/PSELDNets',
            filename='dataset/test360_ov3.zip',
            repo_type='dataset',
        )
        shutil.copy(src, ZIP)
        print(f'ダウンロード完了: {os.path.getsize(ZIP)/1e9:.2f} GB')

    print('解凍中...')
    with zipfile.ZipFile(ZIP) as z:
        z.extractall('datasets/')
    print('解凍完了')
else:
    print(f'既にあります: {DATA_DIR}')

!ls datasets/test360_ov3/

---
## 8. 前処理（FLAC → HDF5 特徴量）

数分かかります。

In [ ]:
if not os.path.exists('_hdf5') or len(os.listdir('_hdf5')) == 0:
    !python src/preproc.py dataset=test360_ov3
else:
    print('既に前処理済み')
    !ls _hdf5/

## 9. 推論実行（test360_ov3 で SELD スコアを確認）

In [ ]:
# 事前確認
checks = [
    ('ckpts/mACCDOA-HTSAT-0.567.ckpt', 'チェックポイント'),
    ('datasets/cls_indices_train.tsv',  'TSV train'),
    ('datasets/cls_indices_test.tsv',   'TSV test'),
    ('datasets/test360_ov3/foa',        'FOA データ'),
    ('_hdf5',                           'HDF5 特徴量'),
]
for path, name in checks:
    ok = os.path.exists(path) and (not os.path.isdir(path) or len(os.listdir(path)) > 0)
    print(f'  [{"OK" if ok else "NG"}] {name}')

In [ ]:
# test360_ov3 だけで評価（ER/F1/LE/LR を表示）。
# 注意: mode=valid は train_dataset も読み込むため、ダウンロードした
#       test360_ov3 のみに train/valid を作り直す。
#       （Hydra の dict は上書きするとマージされ他キーが残るので
#        ~ で削除 → + で再追加する）
!python src/infer.py experiment=synth_maccdoa \
  '~data.train_dataset' '+data.train_dataset={test360_ov3:[fold1_room1,fold1_room2,fold1_room3,fold1_room4,fold1_room5,fold1_room6,fold1_room7,fold1_room8,fold1_room9]}' \
  '~data.valid_dataset' '+data.valid_dataset={test360_ov3:[fold1_room1,fold1_room2,fold1_room3,fold1_room4,fold1_room5,fold1_room6,fold1_room7,fold1_room8,fold1_room9]}' \
  ckpt_path=ckpts/mACCDOA-HTSAT-0.567.ckpt \
  model.kwargs.pretrained_path=null

# 【もし上が Hydra のエラーで落ちる場合のフォールバック（スコアは出ないが確実に通る）】
# !python src/infer.py experiment=synth_maccdoa mode=test \
#   '~data.test_dataset.test1800_ov1' '~data.test_dataset.test900_ov2' \
#   ckpt_path=ckpts/mACCDOA-HTSAT-0.567.ckpt model.kwargs.pretrained_path=null

---
# DCASE2021 ファインチューニング（途中保存＆自動再開）

**切断対策:** チェックポイントを Google Drive に保存し、再接続時は自動で続きから学習します。
データ zip も Drive にキャッシュして再ダウンロードを回避します（Drive を約7.6GB消費）。

切断したら、T4 で再接続 → セル1から順に実行し直せば、学習は last.ckpt から自動で続きから再開します。
（clone / pip / 解凍 / 前処理は毎回再実行が必要。守れるのは「学習時間」です）

## 10. Google Drive をマウント + ワークスペース定義

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
WS = '/content/drive/MyDrive/PSELDNets_workspace'   # ← Drive上の保存先（好きに変えてOK）
os.makedirs(f'{WS}/logs', exist_ok=True)            # 学習チェックポイント置き場
os.makedirs(f'{WS}/zips', exist_ok=True)            # DL zip キャッシュ置き場
print('ワークスペース:', WS)

## 11. データ取得（Drive にキャッシュ → 再DL回避）

In [ ]:
# wget はローカルにDL（FUSE直書きは不安定）→ 完了後 Drive へ移動 → ローカルに symlink
import os, shutil
DCASE    = 'datasets/DCASE2021'
ZIPCACHE = f'{WS}/zips/DCASE2021'     # Drive 上の永続キャッシュ
os.makedirs(DCASE, exist_ok=True)
os.makedirs(ZIPCACHE, exist_ok=True)

BASE  = 'https://zenodo.org/records/5476980/files'
files = ['foa_dev.zip', 'foa_dev.z01', 'foa_eval.zip', 'metadata_dev.zip', 'metadata_eval.zip']

for f in files:
    cache = f'{ZIPCACHE}/{f}'        # Drive（永続）
    local = f'{DCASE}/{f}'           # /content（symlink先）
    if not os.path.exists(cache):
        tmp = f'{DCASE}/{f}.part'
        print(f'DL: {f}')
        !wget -q --show-progress -O "{tmp}" "{BASE}/{f}?download=1"
        shutil.move(tmp, cache)
        print(f'  -> Drive保存: {cache}')
    else:
        print(f'Drive既存（DL不要）: {f}')
    # ローカルに symlink を張り直す（解凍セルは {DCASE}/*.zip をそのまま読める）
    if os.path.lexists(local):
        os.remove(local)
    os.symlink(cache, local)

print('\n完了：datasets/DCASE2021/*.zip は Drive へのリンク')
!ls -lh {DCASE}/

## 12. 解凍・ディレクトリ整理

In [ ]:
DCASE = 'datasets/DCASE2021'

# foa_dev: 分割 zip を結合してから解凍
if not os.path.exists(f'{DCASE}/foa_dev'):
    print('foa_dev を結合・解凍中（数分かかります）...')
    !zip -s 0 {DCASE}/foa_dev.zip --out {DCASE}/foa_dev_agg.zip
    !unzip -q {DCASE}/foa_dev_agg.zip -d {DCASE}/
    !rm {DCASE}/foa_dev_agg.zip
    print('foa_dev 完了')

if not os.path.exists(f'{DCASE}/foa_eval'):
    print('foa_eval を解凍中...')
    !unzip -q {DCASE}/foa_eval.zip -d {DCASE}/

if not os.path.exists(f'{DCASE}/metadata_dev'):
    !unzip -q {DCASE}/metadata_dev.zip -d {DCASE}/

if not os.path.exists(f'{DCASE}/metadata_eval'):
    !unzip -q {DCASE}/metadata_eval.zip -d {DCASE}/

# 解凍後はサブフォルダに wav/csv が入っているので平坦化する
print('ディレクトリ整理中...')
for folder, ext in [('foa_dev', 'wav'), ('foa_eval', 'wav'),
                    ('metadata_dev', 'csv'), ('metadata_eval', 'csv')]:
    path = f'{DCASE}/{folder}'
    !find {path} -mindepth 2 -name "*.{ext}" -exec mv -t {path}/ {{}} + 2>/dev/null || true
    !find {path} -mindepth 1 -maxdepth 1 -type d -exec rm -rf {{}} + 2>/dev/null || true

print('\n整理後のファイル数:')
for d in ['foa_dev', 'foa_eval', 'metadata_dev', 'metadata_eval']:
    n = len(os.listdir(f'{DCASE}/{d}')) if os.path.exists(f'{DCASE}/{d}') else 0
    print(f'  {d}/: {n} ファイル')

## 13. 前処理

In [ ]:
# 訓練データの前処理
!python src/preproc.py dataset=DCASE2021 wav_format=.wav

# 評価データの前処理
!python src/preproc.py dataset=DCASE2021 dataset_type=eval wav_format=.wav

## 14. ファインチューニング（途中保存＆自動再開）

このセルは**毎セッション同じものを実行するだけ**。
初回は配布ckptから開始、切断後は Drive の last.ckpt を検知して自動で続きから。

In [ ]:
# 保存先を「固定の experiment_name + Drive上の log_dir」にするのがポイント:
#  - 毎回同じフォルダに last.ckpt が貯まる
#  - /content が消えても Drive に last.ckpt が残る
#  - last.ckpt があれば自動で続きから、無ければ配布ckptから新規開始
import os

EXP    = 'dcase2021_ft'                 # 固定の実験名（フォルダ名になる）
LOGDIR = f'{WS}/logs'                    # Drive上のログ保存先
LAST   = f'{LOGDIR}/multi_accdoa_HTSAT/runs/{EXP}/checkpoints/last.ckpt'

resume = f'ckpt_path={LAST}' if os.path.exists(LAST) else ''
if resume:
    print('▶ 途中から再開:', LAST)
else:
    print('▶ 新規開始（配布ckpt mACCDOA-HTSAT-0.567 から）')

!python src/train.py experiment=dcase2021/finetune_maccdoa_augmix1 \
    model.batch_size=8 \
    experiment_name={EXP} \
    paths.log_dir={LOGDIR}/ \
    {resume}

## 15. 再開用チェックポイントの確認（任意）

In [ ]:
import os, datetime
LAST = f'{WS}/logs/multi_accdoa_HTSAT/runs/dcase2021_ft/checkpoints/last.ckpt'
if os.path.exists(LAST):
    print('あり:', LAST)
    print('更新:', datetime.datetime.fromtimestamp(os.path.getmtime(LAST)))
    print(f'サイズ: {os.path.getsize(LAST)/1e6:.0f} MB')
else:
    print('まだ無し（未学習）')